# Volume 6, Chapter 11: Measures on Metric Spaces (Python companion)

Exercises first, solutions at the end. Dimension and weak convergence on a computer: box-counting dimensions of the Cantor set and the Sierpiński carpet, Richardson's coastline paradox with the Koch curve and random coastlines, the Kolmogorov–Smirnov test applied to random number generators, and what weak convergence does and does not see. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P11.1 (★★) CS application: box-counting dimension of images.** Render the Cantor set and the Sierpiński carpet as point clouds (or binary images) and count the number $N(\varepsilon)$ of grid boxes of side $\varepsilon=3^{-k}$ that they meet. Fit the slope of $\log N(\varepsilon)$ against $\log\frac1\varepsilon$ and compare with $\frac{\log2}{\log3}$ and $\frac{\log8}{\log3}$ (Theorem 11.11 and Exercise 11.2). What goes wrong with boxes of side $2^{-k}$, and why does the fit still work?

**P11.2 (★★) Richardson's coastline.** Build the Koch curve to depth $7$. Measure its length with rulers of size $3^{-k}$ (walk along the vertices) and its box-counting dimension. Show that the measured length grows like $\varepsilon^{1-D}$ with $D=\frac{\log4}{\log3}$, Richardson's empirical law. Then do the same for a random midpoint-displacement curve (a fractal "coastline") and estimate its dimension.

**P11.3 (★) CS application: testing a random number generator.** Glivenko–Cantelli (Exercise 11.5) says $D_n=\sup_t|F_n(t)-t|\to0$ for uniform samples. Compute $D_n$ for $n=10,\dots,10^6$, check that $\sqrt n\,D_n$ stays of order $1$, compare its law with Kolmogorov's distribution, and use it to catch a bad generator (a linear congruential generator with small modulus, and the generator $x\mapsto\{x+\varphi\}$).

**P11.4 (★) Weak convergence and what it does not see.** (a) Check numerically that $\mathrm{Bin}(n,\frac\lambda n)$ converges weakly to $\mathrm{Poisson}(\lambda)$, measuring the distance $\sup_k|F_n(k)-F(k)|$. (b) Let $\mu_n=(1-\frac1n)\delta_0+\frac1n\delta_n$. Check that $\int f\,d\mu_n\to f(0)$ for bounded continuous $f$, but that the means stay at $1$ (Exercise 11.6). (c) Plot the laws of $\frac{S_n-n/2}{\sqrt{n/4}}$ for coin tosses, discrete for each $n$, and watch them converge weakly to a law with a density.

## Solutions

### P11.1

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def cantor_points(depth=14):
    # left endpoints of the 2^depth intervals of level depth
    x = np.array([0.0])
    for k in range(1, depth + 1):
        x = np.concatenate([x, x + 2 * 3.0 ** -k])
    return x

def carpet_points(depth=7):
    pts = np.zeros((1, 2))
    for k in range(1, depth + 1):
        shifts = np.array([(i, j) for i in range(3) for j in range(3) if (i, j) != (1, 1)]) * 3.0 ** -k
        pts = (pts[:, None, :] + shifts[None, :, :]).reshape(-1, 2)
    return pts + 0.5 * 3.0 ** -depth                       # centres of the smallest squares

def box_count(P, eps):
    P = P.reshape(len(P), -1)
    return len(np.unique(np.floor(P / eps + 1e-9).astype(np.int64), axis=0))

for name, P, kmax, D in [("Cantor", cantor_points() + 0.5 * 3.0 ** -14, 12, np.log(2) / np.log(3)),
                         ("carpet", carpet_points(), 6, np.log(8) / np.log(3))]:
    eps = 3.0 ** -np.arange(1, kmax + 1)
    N = np.array([box_count(P, e) for e in eps])
    slope = np.polyfit(np.log(1 / eps), np.log(N), 1)[0]
    eps2 = 2.0 ** -np.arange(2, int(kmax * 1.5) + 1)
    N2 = np.array([box_count(P, e) for e in eps2])
    slope2 = np.polyfit(np.log(1 / eps2), np.log(N2), 1)[0]
    print(f"{name}: N(3^-k) = {N[:6]} ...  slope {slope:.4f}   dyadic boxes slope {slope2:.4f}   theory {D:.4f}")

With triadic boxes the count is exact, $N(3^{-k})=2^k$ for the Cantor set and $8^k$ for the carpet, and the slope is exactly $\frac{\log2}{\log3}$ or $\frac{\log8}{\log3}$. Dyadic boxes do not fit the self-similarity, so $N(2^{-k})$ wobbles between two multiples of $2^{ks}$, and the points of $\log N$ against $k\log2$ oscillate around a straight line; the least-squares slope still lands close to the theory ($0.641$ and $1.904$ here, against $0.631$ and $1.893$). The *box-counting dimension* $\lim\frac{\log N(\varepsilon)}{\log(1/\varepsilon)}$ is always at least the Hausdorff dimension (a box cover is a cover), and the two agree for self-similar sets like these. It is the dimension that image-analysis software computes, for coastlines, cell boundaries, cracks in materials, and the texture of medical scans, because counting boxes is all a computer can do with pixels.

### P11.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def koch(depth):
    P = np.array([0, 1], dtype=complex)
    w = np.exp(1j * np.pi / 3)
    for _ in range(depth):
        a, b = P[:-1], P[1:]
        d = (b - a) / 3
        Q = np.stack([a, a + d, a + d + d * w, a + 2 * d], axis=1).reshape(-1)
        P = np.append(Q, P[-1])
    return P

def ruler_length(P, step):          # walk along every `step`-th vertex
    Q = P[::step]
    return np.abs(np.diff(Q)).sum()

def box_dim(P, ks, base=3.0):
    X = np.stack([P.real, P.imag], axis=1)
    eps = base ** -np.asarray(ks, dtype=float)
    N = [len(np.unique(np.floor(X / e).astype(np.int64), axis=0)) for e in eps]
    return np.polyfit(np.log(1 / eps), np.log(N), 1)[0]

P = koch(7)
D = np.log(4) / np.log(3)
for k in range(1, 8):
    L = ruler_length(P, 4 ** (7 - k))
    print(f"ruler 3^-{k}: length {L:.4f}   (4/3)^k = {(4/3) ** k:.4f}")
print("box-counting dimension of the Koch curve:", round(box_dim(P, range(2, 7)), 4), "  theory", round(D, 4))

# random midpoint displacement: a fractal coastline with Hurst exponent H
rng = np.random.default_rng(3)
def coastline(levels, H):
    y = np.zeros(2)
    for k in range(levels):
        mid = (y[:-1] + y[1:]) / 2 + rng.standard_normal(len(y) - 1) * 2.0 ** (-(k + 1) * H) * 0.6
        z = np.empty(2 * len(y) - 1); z[0::2] = y; z[1::2] = mid; y = z
    return np.linspace(0, 1, len(y)) + 1j * y
def graph_box_dim(y, ks):            # boxes of side 2^-k over each column of width 2^-k
    n = len(y) - 1; N = []
    for k in ks:
        m = 2 ** k; seg = y[:n].reshape(m, -1); top = np.maximum(seg.max(1), y[n // m::n // m]); bot = np.minimum(seg.min(1), y[n // m::n // m])
        N.append(np.sum(np.floor(top * m) - np.floor(bot * m) + 1))
    return np.polyfit(np.array(ks) * np.log(2), np.log(N), 1)[0]
for H in [0.8, 0.5, 0.3]:
    C = coastline(18, H)
    print(f"H={H}: box dimension {graph_box_dim(C.imag, list(range(4, 13))):.3f}   theory 2-H = {2 - H}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(P.real, P.imag, lw=0.5); ax[0].set_aspect('equal'); ax[0].set_title('Koch curve, depth 7')
C = coastline(14, 0.3); ax[1].plot(C.real, C.imag, lw=0.4); ax[1].set_title('random coastline, H = 0.3')
plt.show()

Each finer ruler finds the curve $\frac43$ times longer: with rulers of size $\varepsilon=3^{-k}$ the length is $(\frac43)^k=\varepsilon^{1-D}$, $D=\frac{\log4}{\log3}\approx1.262$, which is exactly the law Richardson found empirically for the coasts of Britain and Norway (with $D\approx1.25$ and $1.52$), and which Mandelbrot read as a dimension in 1967. The length is $\mathcal H^1$, which is infinite here; the right measure is $\mathcal H^D$, which is finite and positive. The box count finds $1.297$ for the Koch curve, a few hundredths above $1.262$ because only five scales are fitted. The random coastlines built by midpoint displacement with roughness exponent $H$ have dimension $2-H$ in the limit, and the box count recovers it roughly ($1.14$, $1.48$, $1.65$ for $H=0.8,0.5,0.3$): midpoint displacement only approximates fractional Brownian motion, and box counts converge slowly. Rougher curves (smaller $H$) fill more of the plane.

### P11.3

In [ ]:
import numpy as np
from scipy.stats import kstwobign
rng = np.random.default_rng(4)

def ks(u):
    u = np.sort(u); n = len(u); i = np.arange(1, n + 1)
    return max(np.max(i / n - u), np.max(u - (i - 1) / n))

for n in [10, 100, 1000, 10 ** 4, 10 ** 5, 10 ** 6]:
    d = ks(rng.random(n))
    print(f"n={n:>8}: D_n = {d:.5f}   sqrt(n) D_n = {np.sqrt(n) * d:.3f}")
stats = np.array([np.sqrt(1000) * ks(rng.random(1000)) for _ in range(3000)])
for q in [0.5, 0.9, 0.99]:
    print(f"quantile {q}: simulated {np.quantile(stats, q):.3f}   Kolmogorov {kstwobign.ppf(q):.3f}")

def lcg(n, a=65, c=1, m=2 ** 11, seed=1):
    x, out = seed, np.empty(n)
    for i in range(n):
        x = (a * x + c) % m; out[i] = x / m
    return out
phi = (np.sqrt(5) - 1) / 2
for name, u in [("small LCG", lcg(10 ** 5)), ("x + phi mod 1", (np.arange(10 ** 5) * phi) % 1)]:
    n = len(u)
    print(f"{name}: sqrt(n) D_n = {np.sqrt(n) * ks(u):.3f}   (p-value {kstwobign.sf(np.sqrt(n) * ks(u)):.3g})")
    pairs = np.floor(np.stack([u[:-1], u[1:]], 1) * 256).astype(int)
    print(f"   consecutive pairs occupy {len(np.unique(pairs, axis=0))} of 65536 cells of a 256x256 grid")
u = rng.random(10 ** 5); pairs = np.floor(np.stack([u[:-1], u[1:]], 1) * 256).astype(int)
print(f"a good generator: {len(np.unique(pairs, axis=0))} cells   (expected 65536 (1 - e^(-1.53)) = {65536 * (1 - np.exp(-(10**5 - 1) / 65536)):.0f})")

$D_n$ shrinks like $\frac1{\sqrt n}$, and $\sqrt nD_n$ has a limit law, Kolmogorov's distribution (1933), whose quantiles the simulation reproduces closely. This is the Kolmogorov–Smirnov test: a generator whose $\sqrt nD_n$ is far in the tail fails. But the two bad generators *pass* with flying colours: their outputs are too evenly spread, not too unevenly, and $\sqrt nD_n$ is tiny ($0.16$ and $0.008$, where an honest sample has a median of $0.83$). The trap is that the KS test sees only the one-dimensional law, which a weak-convergence statement cannot go beyond. Looking at consecutive pairs exposes them at once: the LCG with modulus $2^{11}$ and $x\mapsto\{x+\varphi\}$ put their pairs on a few lines: the LCG has at most $2^{11}=2048$ distinct pairs and the rotation fills only a sliver of cells, while a good generator fills about $78\%$ of them. A good generator must make the law of $(U_1,\dots,U_d)$ converge to the uniform law on $[0,1]^d$, for every $d$: a statement about the finite-dimensional laws, as in Kolmogorov's extension theorem.

### P11.4

In [ ]:
import numpy as np
from scipy.stats import binom, poisson, norm
import matplotlib.pyplot as plt
lam = 3
k = np.arange(0, 60)
for n in [10, 100, 1000, 10 ** 4]:
    d = np.max(np.abs(binom.cdf(k, n, lam / n) - poisson.cdf(k, lam)))
    print(f"n={n:>6}: sup_k |F_Bin - F_Poisson| = {d:.2e}")
f = lambda x: np.cos(x) / (1 + x ** 2)
for n in [10, 100, 10 ** 4]:
    integral = (1 - 1 / n) * f(0) + f(n) / n
    print(f"n={n:>6}: int f dmu_n = {integral:.6f} (f(0) = 1)   mean = {(1 - 1/n) * 0 + n / n}")
fig, ax = plt.subplots(figsize=(7, 3))
for n in [4, 16, 64]:
    j = np.arange(n + 1); x = (j - n / 2) / np.sqrt(n / 4)
    ax.step(x, binom.cdf(j, n, 0.5), where='post', label=f'n={n}')
t = np.linspace(-3, 3, 400); ax.plot(t, norm.cdf(t), 'k--', label='Gaussian')
ax.legend(); ax.set_xlim(-3, 3); plt.show()
print("sup distance to the Gaussian CDF:",
      [round(np.max(np.abs(binom.cdf(np.arange(n + 1), n, .5) - norm.cdf((np.arange(n + 1) - n / 2) / np.sqrt(n / 4)))), 4) for n in [4, 16, 64, 256, 1024]])

(a) The distance between the distribution functions falls roughly like $\frac1n$ (from $5\cdot10^{-2}$ at $n=10$ to $4.5\cdot10^{-5}$ at $n=10^4$): weak convergence, as in Corollary 11.5, with a rate (Le Cam's inequality gives $\frac{\lambda^2}n$ in total variation). (b) The integrals of the bounded continuous $f$ converge to $f(0)$, yet the mean is $1$ for every $n$: a mass $\frac1n$ at distance $n$ weighs nothing in the weak topology and everything for $x\mapsto x$. (c) Each law is carried by $n+1$ points, so none has a density, and still the distribution functions approach $\Phi$ uniformly, at the rate $\frac1{\sqrt n}$ of the Berry–Esseen theorem (the jumps of size about $\frac{0.8}{\sqrt n}$ make the sup distance about half of that). Weak convergence is the right notion precisely because it allows discrete laws to converge to continuous ones.